In [3]:
!pip install streamlit openai pyngrok requests beautifulsoup4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.8 MB/s eta 0:00:00


In [4]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00


In [5]:
import requests
from bs4 import BeautifulSoup
import gradio as gr
import re
import json
from groq import Groq



In [6]:
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')
print("Key loaded successfully")

Key loaded successfully


In [7]:
#scrapper
def scrape_page(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=10)

        soup = BeautifulSoup(r.text, "html.parser")

        title = soup.title.text.strip() if soup.title else "Untitled Page"

        headings = [h.text.strip() for h in soup.find_all(["h1", "h2"]) if h.text.strip()]
        paragraphs = [p.text.strip() for p in soup.find_all("p") if p.text.strip()]

        return {
            "title": title,
            "headings": headings[:5] if headings else ["No headings found"],
            "content": paragraphs[:3] if paragraphs else ["No content found"]
        }

    except:
        return {
            "title": "Untitled Page",
            "headings": ["Page load failed"],
            "content": ["Unable to fetch page content"]
        }

In [8]:
#cro score
def calculate_score(text):
    score = 30
    text = (text or "").lower()

    if "%" in text:
        score += 15
    if "shop" in text or "buy" in text:
        score += 10
    if "limited" in text:
        score += 10
    if "now" in text:
        score += 10
    if len(text.split()) > 8:
        score += 5

    return min(score, 95)


In [9]:
def generate_cro_insights(ad_text, original_page, ai_output):

    insights = []

    ad = ad_text.lower()

    if "%" in ad or "off" in ad:
        insights.append("Detected discount intent → added urgency-based messaging")

    insights.append("CTA optimized for action-oriented conversion behavior")
    insights.append("Applied scarcity + urgency principles")
    insights.append("Improved readability and conversion focus")

    return insights

In [10]:
# safe jason
def safe_parse(output):

    data = {
        "headline": "",
        "cta": "",
        "highlight": "",
        "changes": "",
        "improved_section": ""
    }

    current_key = None

    for line in output.split("\n"):

        line = line.strip()

        if line.startswith("HEADLINE:"):
            data["headline"] = line.replace("HEADLINE:", "").strip()

        elif line.startswith("CTA:"):
            data["cta"] = line.replace("CTA:", "").strip()

        elif line.startswith("HIGHLIGHT:"):
            data["highlight"] = line.replace("HIGHLIGHT:", "").strip()

        elif line.startswith("CHANGES:"):
            current_key = "changes"
            data["changes"] = line.replace("CHANGES:", "").strip()

        elif line.startswith("SECTION:"):
            data["improved_section"] = line.replace("SECTION:", "").strip()

        else:
            if current_key == "changes":
                data["changes"] += " " + line

    return data

In [11]:
# Clean output
def clean_ai_output(text):
    import re

    # remove markdown
    text = re.sub(r"```json|```", "", text)

    # fix new lines inside JSON strings (major cause of your error)
    text = text.replace("\n", " ")

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [12]:
#core ai logic
def generate_content(ad_text, page_data):

    prompt = f"""
You are a CRO expert.

Improve the EXISTING landing page.

Return EXACT FORMAT:

HEADLINE: ...
CTA: ...
HIGHLIGHT: ...
SECTION: ...
CHANGES: ...

RULES:
- No JSON
- No markdown
- Always return valid output
- If page is weak, still generate content

Ad:
{ad_text}

Page:
{page_data['title']}
{page_data['headings']}
{page_data['content']}
"""

    try:
        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )

        output = res.choices[0].message.content

        data = {
            "headline": "",
            "cta": "",
            "highlight": "",
            "section": "",
            "changes": ""
        }

        current = None

        for line in output.split("\n"):
            line = line.strip()

            if line.startswith("HEADLINE:"):
                data["headline"] = line.replace("HEADLINE:", "").strip()

            elif line.startswith("CTA:"):
                data["cta"] = line.replace("CTA:", "").strip()

            elif line.startswith("HIGHLIGHT:"):
                data["highlight"] = line.replace("HIGHLIGHT:", "").strip()

            elif line.startswith("SECTION:"):
                data["section"] = line.replace("SECTION:", "").strip()

            elif line.startswith("CHANGES:"):
                current = "changes"
                data["changes"] = ""

            else:
                if current == "changes":
                    data["changes"] += line + " "

        # SAFETY FALLBACKS
        if not data["headline"]:
            data["headline"] = "Limited Time Offer"

        if not data["cta"]:
            data["cta"] = "Shop Now"

        if not data["highlight"]:
            data["highlight"] = "Exclusive deal available"

        if not data["section"]:
            data["section"] = data["headline"] + " " + data["highlight"]

        if not data["changes"]:
            data["changes"] = "Auto-generated CRO optimization based on ad intent"

        return data

    except Exception as e:
        return {
            "headline": "Limited Time Offer",
            "cta": "Shop Now",
            "highlight": "Exclusive Discount Available",
            "section": "Fallback optimized section",
            "changes": str(e)
        }

In [13]:
#page view
def build_original_page(page_data):

    return f"""
    <div style="padding:20px;background:white;border-radius:10px">
        <h2>{page_data.get("title")}</h2>
        <p>{" ".join(page_data.get("headings", []))}</p>
        <button style="padding:10px 20px;">Learn More</button>
    </div>
    """

In [14]:
#personalised page
def build_modified_page(data):

    return f"""
    <div style="padding:25px;border-radius:12px;background:{data['theme_color']};color:white;box-shadow:0 4px 20px rgba(0,0,0,0.2)">
        <h2 style="font-size:26px">{data['headline']}</h2>
        <p style="margin:10px 0">{data['highlight']}</p>
        <button style="padding:14px 30px;background:black;color:white;border:none;border-radius:8px;">
            {data['cta']}
        </button>
        <hr>
        <p><b>Improved Section:</b> {data.get('improved_section')}</p>
    </div>
    """

In [15]:
#main logic
def process(ad_text, url):

    page_data = scrape_page(url)
    ai = generate_content(ad_text, page_data)

    # ---------------- SAFETY FIX (VERY IMPORTANT) ----------------
    headings = page_data.get("headings", [])
    content = page_data.get("content", [])

    if not isinstance(headings, list):
        headings = []
    if not isinstance(content, list):
        content = []

    # ---------------- SCORE SAFE CALCULATION ----------------
    original_text = " ".join(headings + content)

    before = calculate_score(original_text)

    ai_text = " ".join([
        ai.get("headline", ""),
        ai.get("cta", ""),
        ai.get("highlight", "")
    ])

    after = calculate_score(ai_text)

    # ---------------- ORIGINAL PAGE ----------------
    original_page = f"""
    <div style="
        padding:20px;
        font-family:Arial;
        border:1px solid #e5e7eb;
        border-radius:12px;
        background:#ffffff;
    ">
        <h2 style="color:#111827;">
            {page_data.get('title', 'Untitled Page')}
        </h2>

        <p style="color:#6b7280;font-size:14px;">
            {' | '.join(headings) if headings else 'No headings found'}
        </p>

        <hr style="margin:15px 0;">

        <p style="color:#374151;line-height:1.6;">
            {' '.join(content[:3]) if content else 'No content available'}
        </p>
    </div>
    """

    # ---------------- ENHANCED PAGE (LIGHT + MODERN CRO UI) ----------------
    optimized_page = f"""
    <div style="
        font-family:Arial;
        border-radius:16px;
        overflow:hidden;
        border:1px solid #e5e7eb;
        background:#ffffff;
    ">

        <!-- URGENCY BANNER -->
        <div style="
            background:#fef3c7;
            color:#92400e;
            text-align:center;
            padding:10px;
            font-size:13px;
            font-weight:bold;
        ">
            LIMITED TIME OFFER — ACT NOW
        </div>

        <!-- HERO SECTION -->
        <div style="
            padding:30px;
            background:#f9fafb;
            text-align:center;
        ">

            <div style="
                display:inline-block;
                background:#fee2e2;
                color:#b91c1c;
                padding:6px 12px;
                border-radius:20px;
                font-size:12px;
                font-weight:bold;
            ">
                SPECIAL OFFER
            </div>

            <h1 style="
                font-size:28px;
                color:#111827;
                margin:15px 0;
            ">
                {ai.get('headline', 'Limited Time Offer')}
            </h1>

            <p style="
                font-size:16px;
                color:#4b5563;
                margin-bottom:20px;
            ">
                {ai.get('highlight', 'Exclusive deal available')}
            </p>

            <button style="
                padding:14px 22px;
                background:#2563eb;
                color:white;
                border:none;
                border-radius:10px;
                font-size:15px;
                font-weight:bold;
                cursor:pointer;
            ">
                {ai.get('cta', 'Shop Now')}
            </button>

            <div style="
                margin-top:10px;
                font-size:12px;
                color:#6b7280;
            ">
                Secure checkout • Limited availability
            </div>
        </div>

        <!-- CRO FEATURES -->
        <div style="padding:20px; background:white;">

            <h3 style="color:#111827;">Why This Works</h3>

            <ul style="color:#4b5563; line-height:1.7;">
                <li>Urgency increases conversion intent</li>
                <li>Clear CTA reduces friction</li>
                <li>Emotion-driven messaging improves engagement</li>
            </ul>

        </div>

        <!-- AI SECTION -->
        <div style="
            padding:20px;
            background:#f9fafb;
            color:#374151;
            font-size:14px;
        ">
            <b>Optimized Section:</b><br><br>
            {ai.get('section', 'Improved marketing copy will appear here')}
        </div>

    </div>
    """

    # ---------------- CRO ANALYSIS ----------------
    cro_analysis = f"""
AI CRO Analysis:

{ai.get('changes', 'No changes generated')}
"""

    # ---------------- FINAL SCORE ----------------
    score = f"{before} → {after}"

    return original_page, optimized_page, cro_analysis, score

In [16]:
#UI
with gr.Blocks(title="AI Landing Page Personalizer") as app:

    gr.Markdown("# AI Landing Page Personalizer")
    gr.Markdown("Enhance landing pages using AI + CRO intelligence")

    ad = gr.Textbox(label="Ad Creative")
    url = gr.Textbox(label="Landing Page URL")

    btn = gr.Button("Generate")

    with gr.Row():
        original = gr.HTML(label="Original Page")
        optimized = gr.HTML(label="Optimized Page")

    cro = gr.Textbox(label="CRO Analysis")
    score = gr.Textbox(label="Conversion Score")

    btn.click(
        process,
        inputs=[ad, url],
        outputs=[original, optimized, cro, score]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59537711def23721ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
